In [1]:
using Random, Distributions, Statistics, Printf, DelimitedFiles, Dates
using LinearAlgebra
using StatsBase
using QuantileRegressions
using Plots

include("functions.jl")
Random.seed!(2025)
const bb = 27 
const aa = 38
const N  = 1462439
const I0 = 1
const S0 = 1316195
const n_iter = 1_000_000

Istar_obs = [3, 7, 10, 4, 14, 35, 92, 98, 216, 374, 434, 417, 447, 275, 151, 83, 67, 48, 37, 29, 42, 46, 53, 73, 87, 111, 123, 124, 99, 135, 106, 74, 39, 13]

tau = length(Istar_obs)

model_tag_sym = :sliding
KMAX_UPPER = 30  
KMAX_fixed = 27

# Fixed output dir
out_dir = "output"
isdir(out_dir) || mkpath(out_dir)

header_cont = []
if model_tag_sym === :memoryless
    global header_cont = ["beta", "alpha", "gamma"]
elseif model_tag_sym === :powerlaw
    global header_cont = ["beta", "alpha", "gamma", "lambda_P"]
elseif model_tag_sym === :exponential
    global header_cont = ["beta", "alpha", "gamma", "lambda_E"]
elseif model_tag_sym === :reciprocal
    global header_cont = ["beta", "alpha", "gamma", "lambda_R"] 
elseif model_tag_sym === :sliding
    global header_cont = ["beta", "alpha", "gamma"]            
else
    error("Unknown model tag: $(model_tag_sym)")
end

c = 2

@info "[$(String(model_tag_sym))_model] Fitting chain $(c) (tau=$tau)"

Random.seed!(2025 + c)
initθ_chain = initθ_for_chain(model_tag_sym) 
t0 = Dates.now()

try
    samples, loglik_aug_vecs =
        mcmc_one_chain_with_Rstar!(Istar_obs, N,S0, I0;
            fit_mech=model_tag_sym,
            n_iter=n_iter,
            initθ=initθ_chain,
            KMAX_UPPER=KMAX_UPPER,
            k_max_fixed = KMAX_fixed)
    if size(samples, 1) != n_iter
        error("Chain $c did not complete all iterations.")
    end

    # Save samples
    samples_filename = "samples_chain_$(c).csv"
    write_csv(joinpath(out_dir, samples_filename), header_cont, samples)

    # Save per-time log-likelihoods (thinned & post-burnin inside mcmc)
    loglik_filename = "loglik_chain_$(c).csv"
    write_csv(joinpath(out_dir, loglik_filename), ["loglik"], hcat(loglik_aug_vecs))
    el = Dates.value(Dates.now() - t0) / 1000
catch err
    el = Dates.value(Dates.now() - t0) / 1000
end

@info "Chain completed -> output dir: $out_dir"


[ Info: [sliding_model] Fitting chain 2 (tau=34)
[ Info: [sliding] iter 1000/1000000 elapsed=3.7s, rate=0.071, mean=[1.045, 0.00029, 1.101], std=[0.0181, 0.000415, 0.0070] [ADAPT]
[ Info: [sliding] iter 2000/1000000 elapsed=6.9s, rate=0.056, mean=[1.200, 0.00021, 1.074], std=[0.1903, 0.000309, 0.0336] [ADAPT]
[ Info: [sliding] iter 3000/1000000 elapsed=9.2s, rate=0.042, mean=[1.435, 0.00017, 1.046], std=[0.3432, 0.000263, 0.0450] [ADAPT]
[ Info: [sliding] iter 4000/1000000 elapsed=11.4s, rate=0.034, mean=[1.607, 0.00015, 1.028], std=[0.3997, 0.000235, 0.0478] [ADAPT]
[ Info: [sliding] iter 5000/1000000 elapsed=13.6s, rate=0.029, mean=[1.713, 0.00013, 1.020], std=[0.4043, 0.000216, 0.0449] [ADAPT]
[ Info: [sliding] iter 6000/1000000 elapsed=15.8s, rate=0.025, mean=[1.782, 0.00012, 1.014], std=[0.3942, 0.000201, 0.0429] [ADAPT]
[ Info: [sliding] iter 7000/1000000 elapsed=18.0s, rate=0.023, mean=[1.834, 0.00012, 1.009], std=[0.3825, 0.000190, 0.0411] [ADAPT]
[ Info: [sliding] iter 8000/10